In [1]:
import json
import numpy as np
import time
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

In [ ]:
import os
import json
import time
from openai import OpenAI

client = OpenAI(
    api_key=" ", 
    base_url="https://"
)

# 输出所有数据

In [3]:
import json
import numpy as np
import time
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

# --- Paths ---
AUTH_VECTORS_JSON = r"output_data/knowledge_with_embeddings.json"
SOCIAL_DATA_JSON = r"output_data/triplets_with_embeddings_ver3.json"
OUTPUT_FILE = r"newresult/triples_ver3_result1.json"

# --- Load Functions ---
def load_json_data(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

print("Loading authoritative knowledge base...")
RAW_KB_VECTORS = load_json_data(AUTH_VECTORS_JSON)

# ===== Build KB Vector Matrix =====
kb_embeddings = []
kb_entities = []

for item in RAW_KB_VECTORS:
    vec = item.get("embedding")
    if vec:
        kb_embeddings.append(vec)
        kb_entities.append(item)

KB_MATRIX = np.array(kb_embeddings)
print("KB matrix shape:", KB_MATRIX.shape)

# --- Vector Retrieval (with subject match check) ---
def get_best_kb_match(target_vec, social_subject, threshold=0.6):
    if target_vec is None:
        return None, None

    target_vec_np = np.array(target_vec).reshape(1, -1)
    sims = cosine_similarity(target_vec_np, KB_MATRIX)[0]
    best_idx = np.argmax(sims)
    best_sim = sims[best_idx]
    best_match = kb_entities[best_idx]

    # Subject mismatch check: if no word overlaps, treat as retrieval failure
    kb_subject = best_match.get("subject", "").lower()
    social_subj = social_subject.lower()

    if not any(word in kb_subject for word in social_subj.split()):
        return None, float(best_sim)

    if best_sim >= threshold:
        return best_match, float(best_sim)

    return None, float(best_sim)

# --- LLM Verification Function ---
def verify_with_llm_and_kb(social_triplet, context_text, kb_evidence, similarity):
    s = social_triplet.get('subject', 'N/A')
    p = social_triplet.get('predicate', 'N/A')
    o = social_triplet.get('object', 'N/A')

    sim_value = similarity if similarity is not None else 0.0

    prompt = f"""
[Role] You are a senior AI Architect specializing in Knowledge Graph Alignment, focusing on the intersection of Symbolic Reasoning and Vector Space Analysis.
[Objective] Compare "Extracted Triples from Short Videos" with "Authoritative Medical Knowledge Base Triples" and determine factual correctness.

[Social Media Claim]
Subject: {s} | Predicate: {p} | Object: {o}

[Cosine Similarity Score] {sim_value:.4f}
[Semantic Alignment Rule] Cosine similarity >= 0.6 indicates semantic alignment,
accounting for medical synonyms (e.g., "Inject" -> "Administer", "Sugar control" -> "Glycemic management").

[Video Context] {context_text if context_text else "N/A"}
[Authoritative KB Evidence] {kb_evidence if kb_evidence else "No direct evidence found in KB."}

[Critical Evaluation Rules]
1. Perform entity normalization before matching.
Treat semantically equivalent medical terms as the same entity.
Examples:
- "type 2 diabetes" = "type ii diabetes mellitus"
- "t2d" = "type 2 diabetes"
Only disregard KB if the subject is clearly unrelated.

2. When evaluating correctness:
- First attempt to match with KB evidence (if semantic alignment exists).
- If no relevant KB evidence is found, use established medical knowledge for judgment.

3. Only mark False if:
- The claim contradicts strong medical consensus, OR
- The claim contradicts reliable evidence, OR
- The claim is medically implausible or unsupported.

4. The relation "associated with" can describe:
- presence of a property
- absence of a property
Examples:
- associated with insulin resistance
- associated with absence of autoantibodies

5. Only treat KB evidence as contradiction if it refers to the SAME attribute or concept.
   If the KB discusses a different attribute, it should NOT be used to reject the claim.

[Task]
1. Status: Classify as "True" or "False".
   - "True": Claim is medically correct, supported by KB evidence or general medical knowledge.
   - "False": Claim directly contradicts a matching KB entry, established medical facts, or is medically implausible.

2. Explanation: Provide a concise reason for your decision, referencing KB match quality explicitly.
   - If Cosine Similarity < 0.6, you MUST identify a Mismatch Reason from:
     * Granularity Mismatch: Triple is too broad or too specific compared to KB.
     * Contextual Shift: Colloquial/metaphorical language drifted in vector space.
     * Logical Inversion: Predicate direction is reversed or negated.
     * Entity Ambiguity: Entity has multiple meanings causing vector drift.
     * Fact Contradiction: Directly contradicts evidence-based medicine.

Return ONLY a JSON object in this exact format:
{{"status": "True" or "False", "explanation": "...", "mismatch_reason": "Category name or null"}}
"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "You are a professional medical knowledge graph verifier. Always return valid JSON only."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0,
            top_p=1
        )
        return json.loads(response.choices[0].message.content)

    except Exception as e:
        return {"status": "Error", "explanation": str(e), "mismatch_reason": None}

# --- Main Program ---
def main():
    print("Loading social media data...")
    social_data = load_json_data(SOCIAL_DATA_JSON)
    print("Total records:", len(social_data))

    final_report = []
    total_triplets = 0
    true_count = 0

    # ===== Group by Video =====
    grouped = defaultdict(list)
    for row in social_data:
        source = f"Video_{row.get('source_row', 'unknown')}"
        grouped[source].append(row)

    print("Total videos:", len(grouped))

    # ===== Main Loop =====
    for i, (source, triplets) in enumerate(grouped.items()):
        print(f"Processing {i+1}/{len(grouped)}: {source}")

        verified_triplets = []
        video_true = 0
        video_total = 0

        for t in triplets:
            target_vec = t.get('embedding')

            # Pass subject into retrieval for match quality check
            kb_match, similarity = get_best_kb_match(target_vec, t.get('subject', ''))

            if kb_match:
                kb_evidence = (
                    f"Subject: {kb_match.get('subject')} | "
                    f"Predicate: {kb_match.get('predicate')} | "
                    f"Object: {kb_match.get('object')} | "
                    f"Similarity: {similarity:.4f}"
                )
            else:
                kb_evidence = None

            verification = verify_with_llm_and_kb(t, "", kb_evidence, similarity)

            # Remove embedding from output
            t_clean = {k: v for k, v in t.items() if k != 'embedding'}
            t_clean.update(verification)

            if kb_match:
                t_clean['kb_alignment_info'] = {
                    "kb_subject": kb_match.get("subject"),
                    "kb_predicate": kb_match.get("predicate"),
                    "kb_object": kb_match.get("object"),
                    "similarity_score": similarity
                }
            else:
                t_clean['kb_alignment_info'] = {
                    "kb_subject": None,
                    "kb_predicate": None,
                    "kb_object": None,
                    "similarity_score": similarity
                }

            video_total += 1
            total_triplets += 1
            if str(verification.get("status", "")).strip().lower() == "true":
                video_true += 1
                true_count += 1

            verified_triplets.append(t_clean)

        video_accuracy = round(video_true / video_total, 4) if video_total > 0 else 0.0

        final_report.append({
            "video_source": source,
            "video_accuracy": video_accuracy,
            "video_true_count": video_true,
            "video_total_count": video_total,
            "verification_results": verified_triplets
        })

        time.sleep(0.05)

    # ===== Overall Accuracy =====
    overall_accuracy = round(true_count / total_triplets, 4) if total_triplets > 0 else 0.0

    output = {
        "overall_summary": {
            "total_triplets": total_triplets,
            "true_count": true_count,
            "false_count": total_triplets - true_count,
            "overall_accuracy": overall_accuracy
        },
        "video_reports": final_report
    }

    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    print("\n===== Verification Complete =====")
    print(f"Total triplets:     {total_triplets}")
    print(f"True count:         {true_count}")
    print(f"False count:        {total_triplets - true_count}")
    print(f"Overall accuracy:   {overall_accuracy:.2%}")
    print(f"Output file:        {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Loading authoritative knowledge base...
KB matrix shape: (4940, 768)
Loading social media data...
Total records: 2833
Total videos: 295
Processing 1/295: Video_0
Processing 2/295: Video_2
Processing 3/295: Video_3
Processing 4/295: Video_4
Processing 5/295: Video_5
Processing 6/295: Video_6
Processing 7/295: Video_7
Processing 8/295: Video_8
Processing 9/295: Video_10
Processing 10/295: Video_11
Processing 11/295: Video_12
Processing 12/295: Video_13
Processing 13/295: Video_14
Processing 14/295: Video_15
Processing 15/295: Video_16
Processing 16/295: Video_17
Processing 17/295: Video_18
Processing 18/295: Video_19
Processing 19/295: Video_20
Processing 20/295: Video_21
Processing 21/295: Video_22
Processing 22/295: Video_23
Processing 23/295: Video_24
Processing 24/295: Video_25
Processing 25/295: Video_26
Processing 26/295: Video_27
Processing 27/295: Video_28
Processing 28/295: Video_29
Processing 29/295: Video_30
Processing 30/295: Video_35
Processing 31/295: Video_36
Processing 32